# Ilera-ATLAS · 05_train: QLoRA fine-tuning of N-ATLaS on a free Kaggle T4

**Runbook Part 8.** One notebook, three runs (change `RUN` in Config):

| Run | Data | When | Account |
|---|---|---|---|
| `runA` | English only | **Tue 6 Oct** | yours |
| `runB` | full mix: en + ha + yo + referral + Aya | **Wed 7 Oct** | yours |
| `runC` | full mix, vignettes and referral cases counted twice | **Wed 7 Oct**, in parallel | a teammate's Kaggle account |

**Always run in Commit mode:** Save Version → Save & Run All (Commit). Kaggle then trains in the
background for up to 12 hours, even with your laptop closed.

**Crash-safe:** every `SAVE_STEPS` steps the latest checkpoint is pushed to a private Hugging Face repo
(`ilera-atlas-ckpt-{RUN}`). If a session dies, commit again: the notebook downloads that checkpoint
and **resumes** instead of starting over.

**After training:** choose the best run with `04_eval.ipynb` (`EVAL_SET = "val"`), never on the benchmark.

**Kaggle:** GPU T4 x2 (one GPU is used) · Internet On · secret `HF_TOKEN` (the teammate uses their own
token, after accepting the N-ATLaS terms and getting write access to your `ilera-data` dataset or a copy).

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"           # <- owner of the ilera-data dataset
OUT_USER = HF_USER                     # account the adapter is pushed to (teammate: their own username)
DATA_REPO = f"{HF_USER}/ilera-data"
BASE_MODEL = "NCAIR1/N-ATLaS"
RUN = "runA"                           # "runA", "runB" or "runC"

PRESETS = {
    "runA": dict(langs=["en"], epochs=2, upweight=1),
    "runB": dict(langs=None, epochs=2, upweight=1),          # None = all languages incl. Aya
    "runC": dict(langs=None, epochs=2, upweight=2),          # vignettes + referral cases x2
}
MAX_LEN = 2048                          # same value as MAX_TOKENS in 03b_build
LORA_R, LORA_ALPHA = 16, 16
LR, WARMUP, WEIGHT_DECAY = 2e-4, 0.05, 0.01
BATCH, GRAD_ACCUM = 2, 4                # effective batch 8; OOM -> BATCH 1, GRAD_ACCUM 8
EVAL_STEPS = SAVE_STEPS = 100
SEED = 3407
MAX_STEPS = -1                          # -1 = full epochs; e.g. 30 for a quick smoke test
RESUME = True                           # continue from the checkpoint on HF if one exists
UPLOAD = True
LOCAL_DIR = None                        # offline testing only

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
# If the next cells fail with an import error: Run > Restart kernel, then Run All again.

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
if UPLOAD or not LOCAL_DIR:
    assert HF_TOKEN, "No HF_TOKEN found: add it as a Kaggle/Colab secret or env var"
    os.environ["HF_TOKEN"] = HF_TOKEN
cfg = PRESETS[RUN]
ADAPTER_REPO = f"{OUT_USER}/ilera-atlas-lora-{RUN}"
CKPT_REPO = f"{OUT_USER}/ilera-atlas-ckpt-{RUN}"
OUT_DIR = f"outputs/{RUN}"
print(RUN, cfg, "->", ADAPTER_REPO)

## 1. Load and prepare the training data

In [ ]:
import collections, json, random

def fetch(path):
    if LOCAL_DIR:
        return os.path.join(LOCAL_DIR, path)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(DATA_REPO, path, repo_type="dataset", token=HF_TOKEN)

def read_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

train_rows = read_jsonl(fetch("train.jsonl"))
val_rows = read_jsonl(fetch("val.jsonl"))
stats = json.load(open(fetch("train_stats.json")))

if cfg["langs"]:
    train_rows = [r for r in train_rows if r["lang"] in cfg["langs"]]
    val_rows = [r for r in val_rows if r["lang"] in cfg["langs"]]
if cfg["upweight"] > 1:
    extra = [r for r in train_rows if r["kind"] in ("vignette", "referral")]
    train_rows = train_rows + extra * (cfg["upweight"] - 1)
random.Random(SEED).shuffle(train_rows)

print(f"{RUN}: train {len(train_rows)} | val {len(val_rows)}")
print("  by lang:", dict(collections.Counter(r["lang"] for r in train_rows)))
print("  by kind:", dict(collections.Counter(r["kind"] for r in train_rows)))
assert len(train_rows) > 100, "training set is suspiciously small: check train.jsonl and RUN"

## 2. Load N-ATLaS in 4-bit and add LoRA adapters
N-ATLaS has 8B parameters. In 4-bit it needs about 6 GB of the T4's 15 GB; LoRA (r = 16 on all
attention and MLP projections) trains about 42M extra weights, roughly 0.5% of the model.

In [ ]:
import torch
from unsloth import FastLanguageModel

assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 x2 in the notebook settings"
print("GPU:", torch.cuda.get_device_name(0))
model, tok = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN,
                                               load_in_4bit=True, token=HF_TOKEN)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)
USE_BF16 = torch.cuda.is_bf16_supported()      # False on T4 -> fp16

## 3. Chat-format the data
Each example is rendered with N-ATLaS's own chat template, the same one used at evaluation and in the
API. The check below makes sure the Llama-3 role markers are present, because step 4 relies on them
to train **only on the assistant's answer**.

In [ ]:
from datasets import Dataset

USER_MARK = "<|start_header_id|>user<|end_header_id|>\n\n"
ASSIST_MARK = "<|start_header_id|>assistant<|end_header_id|>\n\n"

def render(rows):
    return Dataset.from_list([{"text": tok.apply_chat_template(r["messages"], tokenize=False)} for r in rows])

train_ds, val_ds = render(train_rows), render(val_rows)
sample = train_ds[0]["text"]
assert USER_MARK in sample and ASSIST_MARK in sample, (
    "Chat template markers not found; train_on_responses_only would train on the questions too. "
    "Print `sample` and set USER_MARK / ASSIST_MARK to the template's actual role headers.")
lengths = [len(tok(t["text"], add_special_tokens=False).input_ids) for t in train_ds.select(range(min(300, len(train_ds))))]
print(f"sample lengths (tokens): median {sorted(lengths)[len(lengths) // 2]}, max {max(lengths)} (limit {MAX_LEN})")
print(sample[-700:])

## 4. Trainer
Library argument names change between TRL versions (`tokenizer` → `processing_class`,
`max_seq_length` → `max_length`, `evaluation_strategy` → `eval_strategy`), so they are picked
automatically from whatever version is installed. `warmup_ratio` no longer exists in
transformers 5, so the warmup is given as a number of steps (5% of the total).

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

def pick(params, *names):
    return next(n for n in names if n in params)

cfg_params = inspect.signature(SFTConfig.__init__).parameters
n_steps = MAX_STEPS if MAX_STEPS > 0 else -(-len(train_ds) // (BATCH * GRAD_ACCUM)) * cfg["epochs"]
args = {
    "output_dir": OUT_DIR, "dataset_text_field": "text",
    pick(cfg_params, "max_length", "max_seq_length"): MAX_LEN,
    "per_device_train_batch_size": BATCH, "per_device_eval_batch_size": BATCH,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "num_train_epochs": cfg["epochs"], "max_steps": MAX_STEPS,
    "learning_rate": LR, "lr_scheduler_type": "cosine",
    "warmup_steps": max(1, int(WARMUP * n_steps)),   # warmup_ratio was removed in transformers 5
    "weight_decay": WEIGHT_DECAY, "optim": "adamw_8bit",
    "fp16": not USE_BF16, "bf16": USE_BF16,
    "logging_steps": 10,
    pick(cfg_params, "eval_strategy", "evaluation_strategy"): "steps", "eval_steps": EVAL_STEPS,
    "save_strategy": "steps", "save_steps": SAVE_STEPS, "save_total_limit": 2,
    "report_to": "none", "seed": SEED,
}
if UPLOAD:   # push the latest checkpoint to a private repo so a dead session can resume
    args.update(push_to_hub=True, hub_model_id=CKPT_REPO, hub_strategy="checkpoint",
                hub_private_repo=True, hub_token=HF_TOKEN)
tr_params = inspect.signature(SFTTrainer.__init__).parameters
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=SFTConfig(**args),
                     **{pick(tr_params, "processing_class", "tokenizer"): tok})
print(f"{n_steps} optimizer steps (~{n_steps * 2.2 / 60:.1f} h on a T4 is a rough guide)")

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer, instruction_part=USER_MARK, response_part=ASSIST_MARK)
labels = trainer.train_dataset[0]["labels"]
n_trained = sum(l != -100 for l in labels)
assert 0 < n_trained < len(labels), "response masking failed: every token (or none) is trained"
print(f"masking OK: {n_trained} of {len(labels)} tokens of example 0 are trained (the answer only)")

## 5. Train (resumes automatically)

In [ ]:
import glob, shutil

resume = None
if RESUME:
    local = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    if local:
        resume = local[-1]
    elif UPLOAD:
        try:
            from huggingface_hub import snapshot_download
            snap = snapshot_download(CKPT_REPO, token=HF_TOKEN, allow_patterns=["last-checkpoint/*"])
            if os.path.exists(f"{snap}/last-checkpoint/trainer_state.json"):
                resume = f"{OUT_DIR}/checkpoint-resume"
                shutil.copytree(f"{snap}/last-checkpoint", resume, dirs_exist_ok=True)
        except Exception:
            pass
print("resuming from", resume) if resume else print("starting from scratch")

result = trainer.train(resume_from_checkpoint=resume)
final_eval = trainer.evaluate()
print(result.metrics)
print("final validation loss:", round(final_eval["eval_loss"], 4))

## 6. Training curves
Healthy: training loss falls fast in the first ~100 steps, then flattens (often around 0.6 to 1.0);
validation loss falls then flattens. If validation loss **rises** while training loss keeps falling, the
model is overfitting: use 1 epoch, or the earlier checkpoint.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log = trainer.state.log_history
tr = [(e["step"], e["loss"]) for e in log if "loss" in e]
ev = [(e["step"], e["eval_loss"]) for e in log if "eval_loss" in e]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(*zip(*tr), label="train loss", linewidth=1)
if ev:
    ax.plot(*zip(*ev), "o-", label="validation loss")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title(f"Ilera-ATLAS {RUN}"); ax.legend(); ax.grid(alpha=.3)
os.makedirs("results", exist_ok=True)
fig.tight_layout(); fig.savefig(f"results/loss_{RUN}.png", dpi=150)
json.dump(log, open(f"results/train_log_{RUN}.json", "w"), indent=1)
if ev and len(ev) >= 3 and ev[-1][1] > min(v for _, v in ev) * 1.03:
    best = min(ev, key=lambda x: x[1])
    print(f"WARNING: validation loss rose from {best[1]:.3f} (step {best[0]}) to {ev[-1][1]:.3f}: possible overfitting")

## 7. Save the adapter and its model card
The LoRA adapter (~170 MB) is pushed to a private repo. `04_eval.ipynb` loads it directly
(`MODEL = "<user>/ilera-atlas-lora-runB"`): Unsloth fetches N-ATLaS and applies the adapter.
The README carries the licence terms N-ATLaS requires for derivatives.

In [ ]:
import datetime, subprocess, sys
card = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: other
license_name: n-atlas-terms-of-use
tags: [n-atlas, lora, health, nigeria, hausa, yoruba]
---
# Ilera-ATLAS {RUN} (LoRA adapter): Powered by Awarri

LoRA adapter on **{BASE_MODEL}** for primary-health-care decision support aligned with Nigeria's
2024 National Standing Orders for community health workers (CHPRBN / FMoH / NPHCDA).
Decision support for **trained health workers only**; not a diagnosis tool, not for patients.

Attribution: Awarri Technologies and the Federal Ministry of Communications, Innovation and Digital
Economy. Released under the N-ATLaS Terms of Use (same licence for derivatives; 1,000 active-user cap
without a commercial licence).

## Training
- Run: {RUN} {cfg} · examples: {len(train_ds)} train / {len(val_ds)} val
- QLoRA 4-bit, r={LORA_R}, alpha={LORA_ALPHA}, lr={LR}, cosine, warmup {WARMUP}, effective batch {BATCH * GRAD_ACCUM},
  max length {MAX_LEN}, {'bf16' if USE_BF16 else 'fp16'}, seed {SEED}, loss on assistant tokens only
- Final validation loss: {final_eval['eval_loss']:.4f}
- Trained {datetime.date.today()} on {torch.cuda.get_device_name(0)} with Unsloth
- Data: Ilera-QA (Standing Orders Q&A and case vignettes in English, Hausa, Yoruba, human-reviewed)
  plus ~10% Aya Dataset (Apache-2.0) general instructions
"""
open("README_adapter.md", "w", encoding="utf-8").write(card)
subprocess.run(f"{sys.executable} -m pip freeze > results/requirements_train_{RUN}.txt", shell=True)

if UPLOAD:
    from huggingface_hub import HfApi
    model.push_to_hub(ADAPTER_REPO, token=HF_TOKEN, private=True)
    tok.push_to_hub(ADAPTER_REPO, token=HF_TOKEN, private=True)
    api = HfApi(token=HF_TOKEN)
    api.upload_file(path_or_fileobj="README_adapter.md", path_in_repo="README.md", repo_id=ADAPTER_REPO)
    for f in (f"loss_{RUN}.png", f"train_log_{RUN}.json", f"requirements_train_{RUN}.txt"):
        try:
            api.upload_file(path_or_fileobj=f"results/{f}", path_in_repo=f"results/{f}", repo_id=DATA_REPO,
                            repo_type="dataset", commit_message=f"05_train {RUN}: {f}")
        except Exception as e:
            print("could not upload", f, "(", str(e)[:80], ")")
    print(f"adapter: https://huggingface.co/{ADAPTER_REPO}")
else:
    model.save_pretrained(f"{OUT_DIR}/adapter"); tok.save_pretrained(f"{OUT_DIR}/adapter")
    print("saved locally to", f"{OUT_DIR}/adapter")

## Done when
* **Tue 6 Oct:** `ilera-atlas-lora-runA` on Hugging Face, `results/loss_runA.png` looks healthy.
* **Wed 7 Oct:** runB and runC done (two accounts in parallel). Then run `04_eval.ipynb` with
  `EVAL_SET = "val"` for each run (`TAG = "runA"`, `"runB"`, `"runC"`) and choose the run with the best
  `action_ok` and lowest `under_referral`. Only that run goes on to the benchmark (Part 9).

**Troubleshooting:** CUDA out of memory → `BATCH = 1`, `GRAD_ACCUM = 8` (or `MAX_LEN = 1024`).
Loss `nan` or stuck above 2 → `LR = 1e-4`. Session died → commit again (resumes). Import errors after
install → restart the kernel once.